# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


## Dataset yang Digunakan: Telco Customer Churn

**Sumber:** [Kaggle - IBM Sample Data Sets](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

**Deskripsi:**  
Dataset ini berisi informasi pelanggan perusahaan telekomunikasi fiktif (IBM) dan apakah mereka berhenti berlangganan (*churn*) dalam sebulan terakhir.

**Karakteristik Dataset:**
- **Jumlah baris:** 7.043 pelanggan
- **Jumlah kolom:** 21 fitur
- **Tipe masalah:** Binary Classification (prediksi Churn: Yes/No)
- **Target variable:** `Churn`

**Fitur utama:**
- **Demografis:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- **Layanan:** `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- **Akun:** `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Libraries imported successfully!')

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
df = pd.read_csv('../WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape dataset: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
print()
df.head()

In [ ]:
print('=== INFO DATASET ===')
df.info()

In [ ]:
print('=== STATISTIK DESKRIPTIF ===')
df.describe(include='all')

In [ ]:
print('=== DISTRIBUSI TARGET (Churn) ===')
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print(churn_counts)
print()
print(churn_pct.round(2))
print(f'\nChurn rate: {churn_pct["Yes"]:.2f}%')

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# ── 1. Distribusi target variable ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Distribusi Target: Churn', fontsize=14)

sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Jumlah Pelanggan per Kelas Churn', fontsize=14)
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Jumlah')

plt.tight_layout()
plt.show()
print('Dataset tidak seimbang: ~26% pelanggan churn.')

In [ ]:
# ── 2. Churn rate per Contract type ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack()
contract_churn['Yes'].sort_values(ascending=False).plot(
    kind='bar', color='#e74c3c', ax=axes[0])
axes[0].set_title('Churn Rate per Tipe Kontrak', fontsize=13)
axes[0].set_xlabel('Tipe Kontrak')
axes[0].set_ylabel('Churn Rate')
axes[0].tick_params(axis='x', rotation=20)

# ── 3. Churn rate per Internet Service ────────────────────────────────────────
internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack()
internet_churn['Yes'].sort_values(ascending=False).plot(
    kind='bar', color='#3498db', ax=axes[1])
axes[1].set_title('Churn Rate per Internet Service', fontsize=13)
axes[1].set_xlabel('Internet Service')
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Distribusi fitur numerik berdasarkan Churn ─────────────────────────────
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(numeric_cols):
    for churn_val, color, label in [('No', '#2ecc71', 'No Churn'), ('Yes', '#e74c3c', 'Churn')]:
        subset = df[df['Churn'] == churn_val][col]
        # TotalCharges has whitespace entries — convert to numeric first
        subset = pd.to_numeric(subset, errors='coerce').dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label)
    axes[i].set_title(f'Distribusi {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Pengecekan missing values ──────────────────────────────────────────────
print('Missing values per kolom:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Tidak ada missing values langsung.')

# TotalCharges memiliki spasi kosong yang perlu dideteksi
tc_blank = (df['TotalCharges'].str.strip() == '').sum()
print(f'\nTotalCharges rows dengan nilai kosong (spasi): {tc_blank}')

# ── 6. Cek duplikasi ──────────────────────────────────────────────────────────
print(f'\nJumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# ── 7. Korelasi fitur numerik ─────────────────────────────────────────────────
df_num = df[['SeniorCitizen', 'tenure', 'MonthlyCharges']].copy()
df_num['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df_num['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(8, 6))
sns.heatmap(df_num.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=13)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
# ── Step 1: Salin dataframe agar data asli tidak berubah ──────────────────────
df_clean = df.copy()

# ── Step 2: Hapus kolom yang tidak relevan (customerID) ──────────────────────
df_clean.drop(columns=['customerID'], inplace=True)
print(f'Setelah drop customerID: {df_clean.shape}')

# ── Step 3: Menangani missing values ──────────────────────────────────────────
# TotalCharges: 11 baris berisi spasi kosong → konversi ke NaN lalu isi dengan median
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
median_tc = df_clean['TotalCharges'].median()
df_clean['TotalCharges'].fillna(median_tc, inplace=True)
print(f'Missing values setelah perbaikan: {df_clean.isnull().sum().sum()}')

# ── Step 4: Menghapus duplikat ─────────────────────────────────────────────────
before = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
print(f'Baris duplikat dihapus: {before - df_clean.shape[0]}')

In [ ]:
# ── Step 5: Encoding data kategorikal ─────────────────────────────────────────

# 5a. Sederhanakan nilai 'No phone service' dan 'No internet service' menjadi 'No'
service_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
]
for col in service_cols:
    df_clean[col] = df_clean[col].replace({
        'No phone service': 'No',
        'No internet service': 'No',
    })

# 5b. Binary Yes/No → 1/0
binary_cols = [
    'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
]
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'Yes': 1, 'No': 0})

# 5c. Gender → 1/0
df_clean['gender'] = df_clean['gender'].map({'Male': 1, 'Female': 0})

# 5d. Multi-kategori → Label Encoding
le = LabelEncoder()
for col in ['InternetService', 'Contract', 'PaymentMethod']:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# 5e. Target variable Churn → 1/0
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

print('Encoding selesai.')
print(df_clean.dtypes)

In [ ]:
# ── Step 6: Normalisasi fitur numerik ─────────────────────────────────────────
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

print('Fitur numerik setelah standarisasi:')
df_clean[numeric_cols].describe().round(3)

In [ ]:
# ── Step 7: Verifikasi hasil preprocessing ────────────────────────────────────
print(f'Shape akhir: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
print(f'Distribusi Churn:\n{df_clean["Churn"].value_counts()}')
df_clean.head()

In [ ]:
# ── Step 8: Simpan hasil preprocessing ───────────────────────────────────────
import os

output_path = 'telco_churn_preprocessed.csv'
df_clean.to_csv(output_path, index=False)
print(f'Dataset preprocessed disimpan ke: {os.path.abspath(output_path)}')
print(f'Shape: {df_clean.shape}')